# Trích xuất Text Embeddings cho Đồ án Tốt nghiệp (DATN)
Notebook này được thiết kế để chạy trên Kaggle nhằm tận dụng GPU (T4 hoặc P100) để trích xuất đặc trưng văn bản sản phẩm bằng mô hình `SentenceTransformers`.

In [ ]:
# 1. Cài đặt các thư viện cần thiết
!pip install -q polars sentence-transformers pyarrow

In [ ]:
# 2. Import các thư viện
import os
import numpy as np
import polars as pl
import torch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

# Kiểm tra thiết bị phần cứng (GPU/CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

In [ ]:
# 3. Cấu hình đường dẫn
# Hãy thay đổi INPUT_DIR thành đường dẫn chứa dataset của bạn trên Kaggle
INPUT_DIR = "/kaggle/input/datn-stream-subset" 
OUTPUT_DIR = "/kaggle/working"

# Lựa chọn mô hình SentenceTransformer phù hợp
MODEL_NAME = "all-MiniLM-L6-v2"

# Đường dẫn đến file items.parquet
items_path = os.path.join(INPUT_DIR, "items.parquet")
if not os.path.exists(items_path):
    # Fallback cho chạy local hoặc thử nghiệm
    items_path = "../data/items.parquet"
    INPUT_DIR = "../data"
    
print(f"Loading items from: {items_path}")

In [ ]:
# 4. Đọc dữ liệu sản phẩm bằng Polars
df_items = pl.read_parquet(items_path)
print(f"Tổng số sản phẩm: {len(df_items)}")
print("Các cột dữ liệu:", df_items.columns)
df_items.head(3)

In [ ]:
# 5. Tiền xử lý dữ liệu Văn bản
# Gộp Title, Brand, Description và Features thành một chuỗi đại diện
def prepare_text(row):
    parts = []
    if row["title"]:
        parts.append(f"Title: {row['title']}")
    if row["brand"]:
        parts.append(f"Brand: {row['brand']}")
    if row["description"]:
        parts.append(f"Description: {row['description']}")
    if row["features"]:
        parts.append(f"Features: {row['features']}")
    
    text = " | ".join(parts).strip()
    return text if text else "unknown"

# Áp dụng gộp text
df_items = df_items.with_columns([
    pl.col("title").fill_null(""),
    pl.col("brand").fill_null(""),
    pl.col("description").fill_null(""),
    pl.col("features").fill_null(""),
])

# Chuyển đổi sang list các dict để xử lý nhanh
items_list = df_items.select(["item_id", "title", "brand", "description", "features"]).to_dicts()
texts_to_encode = [prepare_text(item) for item in tqdm(items_list, desc="Gộp văn bản sản phẩm")]

print(f"Ví dụ văn bản sản phẩm đầu tiên:\n\n{texts_to_encode[0]}")

In [ ]:
# 6. Khởi tạo mô hình SentenceTransformers
print(f"Đang tải mô hình: {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME, device=device)
print("Đã tải mô hình thành công.")

In [ ]:
# 7. Tiến hành trích xuất Text Embedding bằng GPU
print("Đang trích xuất đặc trưng văn bản...")
embeddings = model.encode(
    texts_to_encode,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True
)
print(f"Kích thước ma trận embeddings: {embeddings.shape}")

In [ ]:
# 8. Lưu kết quả ra file để tải về
emb_output_path = os.path.join(OUTPUT_DIR, "text_embeddings.npy")
np.save(emb_output_path, embeddings)
print(f"Đã lưu ma trận vector nhúng tại: {emb_output_path}")

# Lưu file ánh xạ index -> item_id để sử dụng khi lập index FAISS
df_metadata = pl.DataFrame({
    "index": list(range(len(df_items))),
    "item_id": df_items["item_id"].to_list()
})
meta_output_path = os.path.join(OUTPUT_DIR, "text_embedding_metadata.parquet")
df_metadata.write_parquet(meta_output_path)
print(f"Đã lưu file ánh xạ metadata tại: {meta_output_path}")

In [ ]:
# 9. Kiểm tra nhanh tính đúng đắn của file đã lưu
loaded_embeddings = np.load(emb_output_path)
print(f"Kiểm tra kích thước file tải lại: {loaded_embeddings.shape}")
assert np.allclose(embeddings, loaded_embeddings), "Dữ liệu lưu bị lỗi!"